# Model Training - Water Potability

Mô hình sử dụng:
- Logistic Regression
- Support Vector Machine (SVM)

Quy trình:
- Median Imputation
- StandardScaler
- Train/Test = 80/20
- 5-Fold Stratified Cross Validation
- Hyperparameter Tuning bằng GridSearchCV

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.exceptions import UndefinedMetricWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC


warnings.filterwarnings(
    "ignore",
    category=UndefinedMetricWarning
)


# Tìm dataset
possible_paths = [
    Path("../data/water_potability.csv"),
    Path("ai-models/data/water_potability.csv"),
    Path("../ai-models/data/water_potability.csv"),
]

data_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Không tìm thấy water_potability.csv"
    )


# Xác định thư mục project
project_root = data_path.resolve().parents[2]

src_path = (
    project_root
    / "ai-models"
    / "src"
)

if str(src_path) not in sys.path:
    sys.path.insert(
        0,
        str(src_path)
    )


from preprocess import (
    load_dataset,
    split_features_target,
    split_train_test,
    build_scaled_preprocessor,
    RANDOM_STATE,
)


# Load dữ liệu
df = load_dataset(data_path)

X, y = split_features_target(df)

X_train, X_test, y_train, y_test = (
    split_train_test(X, y)
)


print("Dataset:", df.shape)

print(
    "Train:",
    X_train.shape
)

print(
    "Test:",
    X_test.shape
)

print(
    "Train/Test overlap:",
    len(
        X_train.index.intersection(
            X_test.index
        )
    )
)

Dataset: (3276, 10)
Train: (2620, 9)
Test: (656, 9)
Train/Test overlap: 0


## 1. Cross Validation Setup

In [2]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}


def cv_summary(results):

    rows = []

    for metric in scoring_metrics:

        values = results[
            f"test_{metric}"
        ]

        rows.append({
            "Metric": (
                metric.upper()
                if metric != "roc_auc"
                else "ROC_AUC"
            ),
            "Mean": np.mean(values),
            "Std": np.std(values),
        })

    return pd.DataFrame(rows)


def train_validation_summary(results):

    rows = []

    for metric in scoring_metrics:

        train_mean = np.mean(
            results[
                f"train_{metric}"
            ]
        )

        validation_mean = np.mean(
            results[
                f"test_{metric}"
            ]
        )

        rows.append({
            "Metric": (
                metric.upper()
                if metric != "roc_auc"
                else "ROC_AUC"
            ),

            "Train_Mean":
                train_mean,

            "Validation_Mean":
                validation_mean,

            "Gap":
                train_mean
                - validation_mean,
        })

    return pd.DataFrame(rows)

## 2. Logistic Regression - Baseline

In [3]:
logistic_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),

    (
        "classifier",
        LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=2000,
        )
    ),
])


logistic_cv_results = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True,
)


print(
    "Logistic Regression Baseline"
)

print(
    cv_summary(
        logistic_cv_results
    ).round(4)
)

print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        logistic_cv_results
    ).round(4)
)

Logistic Regression Baseline
      Metric    Mean     Std
0   ACCURACY  0.6099  0.0009
1  PRECISION  0.0000  0.0000
2     RECALL  0.0000  0.0000
3         F1  0.0000  0.0000
4    ROC_AUC  0.4774  0.0177

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.6103           0.6099  0.0004
1  PRECISION      0.4000           0.0000  0.4000
2     RECALL      0.0010           0.0000  0.0010
3         F1      0.0020           0.0000  0.0020
4    ROC_AUC      0.5216           0.4774  0.0442


Logistic Regression baseline có xu hướng dự đoán hầu hết mẫu về lớp 0.

Do dữ liệu mất cân bằng và F1 của lớp Potable thấp,
mô hình tiếp tục được tuning bằng F1 Score.

## 3. Logistic Regression - Hyperparameter Tuning

In [4]:
logistic_param_grid = {

    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
    ],

    "classifier__class_weight": [
        None,
        "balanced",
    ],

    "classifier__solver": [
        "liblinear",
        "lbfgs",
    ],
}


logistic_grid_search = GridSearchCV(

    estimator=logistic_pipeline,

    param_grid=logistic_param_grid,

    scoring="f1",

    cv=cv_strategy,

    n_jobs=-1,

    verbose=0,

    return_train_score=True,
)


logistic_grid_search.fit(
    X_train,
    y_train
)


best_logistic_pipeline = (
    logistic_grid_search
    .best_estimator_
)


best_logistic_cv_results = (
    cross_validate(

        estimator=
            best_logistic_pipeline,

        X=X_train,

        y=y_train,

        cv=cv_strategy,

        scoring=
            scoring_metrics,

        return_train_score=True,
    )
)


print(
    "Best parameters:"
)

print(
    logistic_grid_search
    .best_params_
)


print(
    "\nBest CV F1:",
    round(
        logistic_grid_search
        .best_score_,
        4
    )
)


print(
    "\nCV sau tuning:"
)

print(
    cv_summary(
        best_logistic_cv_results
    ).round(4)
)


print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        best_logistic_cv_results
    ).round(4)
)

Best parameters:
{'classifier__C': 0.01, 'classifier__class_weight': 'balanced', 'classifier__solver': 'liblinear'}

Best CV F1: 0.4183

CV sau tuning:
      Metric    Mean     Std
0   ACCURACY  0.4962  0.0166
1  PRECISION  0.3802  0.0199
2     RECALL  0.4658  0.0456
3         F1  0.4183  0.0288
4    ROC_AUC  0.4766  0.0177

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.5147           0.4962  0.0185
1  PRECISION      0.4021           0.3802  0.0218
2     RECALL      0.5012           0.4658  0.0354
3         F1      0.4462           0.4183  0.0279
4    ROC_AUC      0.5220           0.4766  0.0454


## 4. Support Vector Machine - Baseline

In [5]:
svm_baseline_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),

    (
        "classifier",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
        )
    ),
])


svm_cv_results = cross_validate(

    estimator=
        svm_baseline_pipeline,

    X=X_train,

    y=y_train,

    cv=cv_strategy,

    scoring=
        scoring_metrics,

    return_train_score=True,
)


print(
    "SVM Baseline"
)

print(
    cv_summary(
        svm_cv_results
    ).round(4)
)


print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        svm_cv_results
    ).round(4)
)

SVM Baseline
      Metric    Mean     Std
0   ACCURACY  0.6798  0.0112
1  PRECISION  0.7190  0.0236
2     RECALL  0.2935  0.0320
3         F1  0.4161  0.0344
4    ROC_AUC  0.7019  0.0179

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.7409           0.6798  0.0612
1  PRECISION      0.8683           0.7190  0.1493
2     RECALL      0.3960           0.2935  0.1025
3         F1      0.5438           0.4161  0.1277
4    ROC_AUC      0.8225           0.7019  0.1206


SVM baseline có Accuracy và ROC-AUC tốt hơn Logistic Regression.

Tuy nhiên Recall còn thấp, vì vậy tiếp tục tuning các tham số:
- C
- gamma
- class_weight

## 5. Support Vector Machine - Hyperparameter Tuning

In [6]:
svm_tuning_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),

    (
        "classifier",
        SVC()
    ),
])


svm_param_grid = [
    {
        "classifier__kernel": [
            "rbf",
        ],

        "classifier__C": [
            0.1,
            1.0,
            10.0,
        ],

        "classifier__gamma": [
            "scale",
            0.01,
            0.1,
            1.0,
        ],

        "classifier__class_weight": [
            None,
            "balanced",
        ],
    },

    {
        "classifier__kernel": [
            "linear",
        ],

        "classifier__C": [
            0.1,
            1.0,
            10.0,
        ],

        "classifier__class_weight": [
            None,
            "balanced",
        ],
    },
]


svm_grid_search = GridSearchCV(

    estimator=
        svm_tuning_pipeline,

    param_grid=
        svm_param_grid,

    scoring="f1",

    cv=cv_strategy,

    n_jobs=-1,

    verbose=0,

    return_train_score=True,
)


svm_grid_search.fit(
    X_train,
    y_train
)


best_svm_pipeline = (
    svm_grid_search
    .best_estimator_
)


best_svm_cv_results = (
    cross_validate(

        estimator=
            best_svm_pipeline,

        X=X_train,

        y=y_train,

        cv=cv_strategy,

        scoring=
            scoring_metrics,

        return_train_score=True,
    )
)


print(
    "Best parameters:"
)

print(
    svm_grid_search
    .best_params_
)


print(
    "\nBest CV F1:",
    round(
        svm_grid_search
        .best_score_,
        4
    )
)


print(
    "\nCV sau tuning:"
)

print(
    cv_summary(
        best_svm_cv_results
    ).round(4)
)


print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        best_svm_cv_results
    ).round(4)
)

Best parameters:
{'classifier__C': 1.0, 'classifier__class_weight': 'balanced', 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}

Best CV F1: 0.5704

CV sau tuning:
      Metric    Mean     Std
0   ACCURACY  0.6668  0.0147
1  PRECISION  0.5733  0.0173
2     RECALL  0.5694  0.0523
3         F1  0.5704  0.0299
4    ROC_AUC  0.7036  0.0217

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.7607           0.6668  0.0939
1  PRECISION      0.6893           0.5733  0.1160
2     RECALL      0.7038           0.5694  0.1343
3         F1      0.6964           0.5704  0.1261
4    ROC_AUC      0.8242           0.7036  0.1206


### Nhận xét Hyperparameter Tuning - SVM

Kết quả GridSearchCV lựa chọn cấu hình:

- kernel = rbf
- C = 1.0
- gamma = scale
- class_weight = balanced
- Best CV F1 = 0.5704

Ảnh hưởng của các tham số:

- **C** kiểm soát mức phạt đối với các mẫu bị phân loại sai. C quá lớn có thể làm mô hình khớp mạnh với dữ liệu huấn luyện, trong khi C quá nhỏ tạo biên phân lớp mềm hơn.
- **kernel** quyết định dạng ranh giới phân lớp. Kết quả thực nghiệm cho thấy RBF phù hợp hơn Linear trong các cấu hình đã thử.
- **gamma** kiểm soát phạm vi ảnh hưởng của từng điểm dữ liệu đối với RBF kernel. Giá trị `scale` cho kết quả tốt nhất trong tập tham số đã kiểm tra.
- **class_weight = balanced** giúp mô hình chú ý nhiều hơn đến lớp Potability = 1 trong dữ liệu mất cân bằng.

Sau tuning, SVM đạt:

- CV F1 = 0.5704 ± 0.0299
- CV ROC-AUC = 0.7036 ± 0.0217
- Train Score = 0.7538
- Training Time ≈ 1.1512 giây
- Prediction Time ≈ 0.0777 giây

Candidate SVM cuối được bật `probability=True` để hỗ trợ `predict_proba()` cho ứng dụng. Việc tính xác suất làm tăng chi phí huấn luyện và dự đoán so với cấu hình không yêu cầu probability.

In [7]:
from sklearn.base import clone


# Tạo candidate SVM cuối từ cấu hình tốt nhất
# và bật probability để phục vụ ứng dụng
best_svm_pipeline = clone(
    svm_grid_search.best_estimator_
)

best_svm_pipeline.set_params(
    classifier__probability=True
)

# Fit lại chỉ trên Training Set
best_svm_pipeline.fit(
    X_train,
    y_train
)

print(
    "SVM candidate cuối:"
)

print(
    best_svm_pipeline
    .named_steps["classifier"]
)

print(
    "\nprobability =",
    best_svm_pipeline
    .named_steps["classifier"]
    .probability
)

f:\4\Hoc may co ban\WaterQualityProject\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM candidate cuối:
SVC(class_weight='balanced', probability=True)

probability = True


In [8]:
from time import perf_counter
from sklearn.base import clone


def measure_model_performance(
    model,
    X_train,
    y_train,
    X_test
):
    candidate = clone(model)

    start_train = perf_counter()

    candidate.fit(
        X_train,
        y_train
    )

    train_time = (
        perf_counter()
        - start_train
    )

    train_score = candidate.score(
        X_train,
        y_train
    )

    start_predict = perf_counter()

    candidate.predict(
        X_test
    )

    prediction_time = (
        perf_counter()
        - start_predict
    )

    return {
        "Train_Score": train_score,
        "Training_Time_s": train_time,
        "Prediction_Time_s": prediction_time,
    }


logistic_performance = (
    measure_model_performance(
        best_logistic_pipeline,
        X_train,
        y_train,
        X_test
    )
)

svm_performance = (
    measure_model_performance(
        best_svm_pipeline,
        X_train,
        y_train,
        X_test
    )
)


print(
    "Logistic Regression:"
)

print(
    logistic_performance
)

print(
    "\nSVM:"
)

print(
    svm_performance
)

f:\4\Hoc may co ban\WaterQualityProject\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Logistic Regression:
{'Train_Score': 0.5190839694656488, 'Training_Time_s': 0.01286439999967115, 'Prediction_Time_s': 0.0037394000028143637}

SVM:
{'Train_Score': 0.7538167938931297, 'Training_Time_s': 1.1827575000024808, 'Prediction_Time_s': 0.09017920000042068}


In [9]:
tv2_model_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Train_Score": logistic_performance[
            "Train_Score"
        ],
        "CV_F1_Mean": np.mean(
            best_logistic_cv_results[
                "test_f1"
            ]
        ),
        "CV_F1_Std": np.std(
            best_logistic_cv_results[
                "test_f1"
            ]
        ),
        "CV_ROC_AUC_Mean": np.mean(
            best_logistic_cv_results[
                "test_roc_auc"
            ]
        ),
        "CV_ROC_AUC_Std": np.std(
            best_logistic_cv_results[
                "test_roc_auc"
            ]
        ),
        "Training_Time_s": logistic_performance[
            "Training_Time_s"
        ],
        "Prediction_Time_s": logistic_performance[
            "Prediction_Time_s"
        ],
    },

    {
        "Model": "SVM",
        "Train_Score": svm_performance[
            "Train_Score"
        ],
        "CV_F1_Mean": np.mean(
            best_svm_cv_results[
                "test_f1"
            ]
        ),
        "CV_F1_Std": np.std(
            best_svm_cv_results[
                "test_f1"
            ]
        ),
        "CV_ROC_AUC_Mean": np.mean(
            best_svm_cv_results[
                "test_roc_auc"
            ]
        ),
        "CV_ROC_AUC_Std": np.std(
            best_svm_cv_results[
                "test_roc_auc"
            ]
        ),
        "Training_Time_s": svm_performance[
            "Training_Time_s"
        ],
        "Prediction_Time_s": svm_performance[
            "Prediction_Time_s"
        ],
    },
])


print(
    tv2_model_summary.round(4)
)

                 Model  Train_Score  CV_F1_Mean  CV_F1_Std  CV_ROC_AUC_Mean  \
0  Logistic Regression       0.5191      0.4183     0.0288           0.4766   
1                  SVM       0.7538      0.5704     0.0299           0.7036   

   CV_ROC_AUC_Std  Training_Time_s  Prediction_Time_s  
0          0.0177           0.0129             0.0037  
1          0.0217           1.1828             0.0902  


In [10]:
training_comparison = pd.DataFrame([
    {
        "Model":
            "Logistic Baseline",

        "Accuracy":
            np.mean(
                logistic_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                logistic_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                logistic_cv_results[
                    "test_roc_auc"
                ]
            ),
    },

    {
        "Model":
            "Logistic Tuned",

        "Accuracy":
            np.mean(
                best_logistic_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                best_logistic_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                best_logistic_cv_results[
                    "test_roc_auc"
                ]
            ),
    },

    {
        "Model":
            "SVM Baseline",

        "Accuracy":
            np.mean(
                svm_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                svm_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                svm_cv_results[
                    "test_roc_auc"
                ]
            ),
    },

    {
        "Model":
            "SVM Tuned",

        "Accuracy":
            np.mean(
                best_svm_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                best_svm_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                best_svm_cv_results[
                    "test_roc_auc"
                ]
            ),
    },
])


print(
    training_comparison.round(4)
)

               Model  Accuracy      F1  ROC_AUC
0  Logistic Baseline    0.6099  0.0000   0.4774
1     Logistic Tuned    0.4962  0.4183   0.4766
2       SVM Baseline    0.6798  0.4161   0.7019
3          SVM Tuned    0.6668  0.5704   0.7036


## Kết luận Training

SVM sau tuning đạt kết quả Cross Validation tốt hơn
Logistic Regression, đặc biệt ở F1 và ROC-AUC.

Cấu hình SVM được chọn:

- kernel = rbf
- C = 1.0
- gamma = scale
- class_weight = balanced

Test set chưa được sử dụng trong quá trình tuning.
Việc đánh giá cuối cùng sẽ thực hiện trong `04_evaluate.ipynb`.

In [11]:
import joblib


# Thư mục lưu candidate models
models_dir = (
    project_root
    / "ai-models"
    / "models"
)

models_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Lưu Logistic Regression đã tuning
logistic_model_path = (
    models_dir
    / "logistic_regression.joblib"
)

joblib.dump(
    best_logistic_pipeline,
    logistic_model_path
)


print(
    "Đã lưu Logistic Regression tại:"
)

print(
    logistic_model_path
)

Đã lưu Logistic Regression tại:
F:\4\Hoc may co ban\WaterQualityProject\ai-models\models\logistic_regression.joblib


In [12]:
# Lưu SVM đã tuning
svm_model_path = (
    models_dir
    / "svm.joblib"
)

joblib.dump(
    best_svm_pipeline,
    svm_model_path
)


print(
    "Đã lưu SVM tại:"
)

print(
    svm_model_path
)

Đã lưu SVM tại:
F:\4\Hoc may co ban\WaterQualityProject\ai-models\models\svm.joblib


In [13]:


loaded_logistic_model = joblib.load(
    models_dir / "logistic_regression.joblib"
)

loaded_svm_model = joblib.load(
    models_dir / "svm.joblib"
)



sample_X = X_train.head(5)


logistic_predictions = (
    loaded_logistic_model.predict(sample_X)
)

svm_predictions = (
    loaded_svm_model.predict(sample_X)
)

svm_probabilities = (
    loaded_svm_model.predict_proba(sample_X)
)

print(
    "SVM probabilities:"
)

print(
    svm_probabilities
)

print(
    "\nSVM probability enabled:",
    loaded_svm_model
    .named_steps["classifier"]
    .probability
)

print(
    "Logistic Regression predictions:",
    logistic_predictions
)

print(
    "SVM predictions:",
    svm_predictions
)

print(
    "\nKiểm tra load model: THÀNH CÔNG"
)

SVM probabilities:
[[0.56315357 0.43684643]
 [0.25529236 0.74470764]
 [0.69176851 0.30823149]
 [0.56168847 0.43831153]
 [0.81063921 0.18936079]]

SVM probability enabled: True
Logistic Regression predictions: [0 0 1 0 0]
SVM predictions: [1 1 0 1 0]

Kiểm tra load model: THÀNH CÔNG
